# 05 — Model Training (UPGRADED)
Upgrades:
- **4 classes**: planet / eclipsing_binary / false_positive / planet_candidate
- Rule-based pseudo-labels from science data fill the missing classes for ML training
- **Ensemble**: RF + XGBoost averaged predictions
- **BIC score** as additional feature (included in FEATURE_COLS)
- **LOOCV** when n < 20 for honest evaluation (avoids train==test inflation)
- **Cross-validation** for reliable accuracy estimate

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import os, pickle, warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, learning_curve)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, precision_recall_fscore_support)
from sklearn.pipeline import Pipeline
import xgboost as xgb

print('Imports OK!')

## 2. Load Features

In [14]:
# Corrupted stars — exclude from all analysis
EXCLUDE_TICS = ['261203535']

In [15]:
LABELED_FEAT = '../outputs/labeled_features.csv'
SCIENCE_FEAT = '../outputs/features.csv'
MODELS_DIR   = '../models/'
os.makedirs(MODELS_DIR, exist_ok=True)

science_df = pd.read_csv(SCIENCE_FEAT)
science_df['tic_id'] = science_df['tic_id'].astype(str).str.replace('.0','', regex=False)
print(f'Science samples : {len(science_df)}')

if os.path.exists(LABELED_FEAT):
    labeled_df = pd.read_csv(LABELED_FEAT)
    labeled_df['tic_id'] = labeled_df['tic_id'].astype(str).str.replace('.0','', regex=False)
    print(f'Labeled samples : {len(labeled_df)}')
    print(labeled_df['label'].value_counts())
else:
    labeled_df = pd.DataFrame()
    print('No labeled features — rule-based only')

Science samples : 14
Labeled samples : 15
label
planet              10
eclipsing_binary     5
Name: count, dtype: int64


## 3. NEW — BIC Score Feature

In [ ]:
def compute_bic_score(snr, n_points, n_params_transit=5, n_params_flat=1):
    """
    Delta BIC between transit model and flat model.
    >10 = transit strongly preferred. <0 = flat model preferred.
    """
    if snr <= 0 or n_points <= 0:
        return 0.0
    log_likelihood_ratio = 0.5 * snr**2
    delta_bic = 2 * log_likelihood_ratio - (n_params_transit - n_params_flat) * np.log(n_points)
    return float(delta_bic)


# Add BIC to science features
science_df['delta_bic'] = science_df.apply(
    lambda r: compute_bic_score(r.get('bls_snr_raw', r['snr']), 18000), axis=1
)
print('BIC scores computed for science data')
print(science_df[['tic_id', 'snr', 'delta_bic']].head(10).to_string())

# Add to labeled too
if len(labeled_df) > 0:
    labeled_df['delta_bic'] = labeled_df.apply(
        lambda r: compute_bic_score(r.get('bls_snr_raw', r['snr']), 18000), axis=1
    )

## 4. Define Feature Columns

In [ ]:
# All possible features — use what's available
ALL_FEATURES = [
    # Original features
    'period_days', 'duration_hours', 'depth_ppm', 'snr', 'bls_power',
    'odd_even_diff', 'secondary_depth', 'transit_skewness', 'transit_kurtosis',
    'flux_std', 'flux_range', 'outlier_frac', 'depth_over_std', 'period_over_dur',
    'in_out_scatter_ratio',
    # NEW features
    'tls_snr', 'tls_sde', 'tls_fap', 'tls_rp_rs', 'tls_odd_even',
    'acf_period', 'acf_strength', 'acf_period_match',
    'ls_period', 'ls_power', 'ls_fap', 'ls_period_match',
    'cdpp_1h', 'cdpp_2h', 'cdpp_6h',
    'both_agree',
    'delta_bic',  # BIC score feature (computed in section 3)
]

FEATURE_COLS = [c for c in ALL_FEATURES if c in science_df.columns]
print(f'Using {len(FEATURE_COLS)} features:')
for c in FEATURE_COLS:
    print(f'  {c}')

## 5. Rule-Based Classification (fallback)

In [ ]:
def rule_based_classify(row):
    snr           = row.get('bls_snr_raw', row.get('snr', 0))
    depth_ppm     = row.get('depth_ppm', 0)
    odd_even_diff = row.get('odd_even_diff', 0)
    sec_depth     = row.get('secondary_depth', 0)
    period        = row.get('period_days', 0)
    duration_h    = row.get('duration_hours', 0)
    tls_fap       = row.get('tls_fap', 1.0)
    delta_bic     = row.get('delta_bic', 0)
    acf_match     = row.get('acf_period_match', 0)
    ls_match      = row.get('ls_period_match', 0)
    tls_rp_rs     = row.get('tls_rp_rs', 0)

    # Noise / no signal
    if snr < 5 and tls_fap > 0.05:
        return 'noise', 0.90, 'Low SNR + high FAP'

    # Stellar variability (ACF or LS period matches transit period)
    if acf_match or ls_match:
        return 'false_positive', 0.80, 'Stellar rotation/variability matches transit period'

    # Eclipsing binary indicators
    if odd_even_diff > 0.005:
        return 'eclipsing_binary', 0.88, 'Large odd-even depth difference'
    if sec_depth > 0.0005:
        return 'eclipsing_binary', 0.82, 'Secondary eclipse detected'
    if depth_ppm > 500_000:                       # rp_rs > 0.7 — must be EB
        return 'eclipsing_binary', 0.90, 'Depth too large for planet (>50% flux drop)'
    if tls_rp_rs > 0.5:
        return 'eclipsing_binary', 0.85, 'Rp/Rs > 0.5 indicates eclipsing binary'

    # Strong planet candidate
    if snr >= 7 and tls_fap < 0.01 and delta_bic > 10:
        if depth_ppm < 100_000 and odd_even_diff < 0.005:
            return 'planet', 0.85, 'High SNR + low FAP + high BIC'

    # Marginal planet candidate
    if snr >= 5 and depth_ppm < 100_000 and odd_even_diff < 0.01:
        return 'planet_candidate', 0.65, 'Marginal planet signal'

    return 'unknown', 0.40, 'Ambiguous signal'


results = science_df.apply(
    lambda row: pd.Series(
        rule_based_classify(row),
        index=['rule_label', 'rule_confidence', 'rule_reason']
    ), axis=1
)
science_df = pd.concat([science_df, results], axis=1)
print('Rule-based classification:')
print(science_df[['tic_id', 'rule_label', 'rule_reason']].to_string())

## 6. Train Ensemble Classifier (RF + XGBoost)

In [ ]:
ml_available = len(labeled_df) >= 10

if not ml_available:
    print('Not enough labeled data — using rule-based only')
else:
    feat_cols = [c for c in FEATURE_COLS if c in labeled_df.columns]
    print(f'Using {len(feat_cols)} features  (delta_bic included: {"delta_bic" in feat_cols})')

    # ── Extend labeled set with pseudo-labels for missing classes ────
    # labeled_df only has planet / eclipsing_binary.
    # Science data already has rule-based labels — use high-confidence ones
    # to give the ML model examples of false_positive and planet_candidate.
    PSEUDO_LABEL_MAP = {
        'false_positive'  : 'false_positive',
        'planet_candidate': 'planet_candidate',
        'noise'           : 'false_positive',   # noise (no transit) = false positive
    }
    PSEUDO_CONF_MIN = 0.70

    pseudo_rows = []
    for _, row in science_df.iterrows():
        rl = row.get('rule_label', '')
        rc = row.get('rule_confidence', 0)
        if rl in PSEUDO_LABEL_MAP and rc >= PSEUDO_CONF_MIN:
            new_row = {c: row.get(c, 0) for c in feat_cols}
            new_row['label'] = PSEUDO_LABEL_MAP[rl]
            pseudo_rows.append(new_row)

    if pseudo_rows:
        pseudo_df   = pd.DataFrame(pseudo_rows)
        labeled_aug = pd.concat(
            [labeled_df[feat_cols + ['label']], pseudo_df],
            ignore_index=True
        )
        print(f'\nPseudo-labeled samples added : {len(pseudo_rows)}')
    else:
        labeled_aug = labeled_df[feat_cols + ['label']].copy()
        print('\nNo high-confidence pseudo-labels found — staying with available classes')

    print(f'Training set ({len(labeled_aug)} samples):')
    print(labeled_aug['label'].value_counts())

    X = labeled_aug[feat_cols].fillna(0).values
    y = labeled_aug['label'].values

    le    = LabelEncoder()
    y_enc = le.fit_transform(y)
    print(f'\nML Classes : {le.classes_}')
    print(f'Counts     : {dict(zip(le.classes_, np.bincount(y_enc)))}')
    print(f'Samples    : {len(X)}  |  Features : {len(feat_cols)}')

    if len(X) >= 20:
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y_enc, test_size=0.2, random_state=42,
            stratify=y_enc if len(np.unique(y_enc)) > 1 else None
        )
        print(f'Train: {len(X_tr)}  Test: {len(X_te)}')
    else:
        X_tr, X_te, y_tr, y_te = X, X, y_enc, y_enc
        print(f'⚠️  <20 samples — LOOCV will be used for honest evaluation (not train=test)')

    # Random Forest
    rf_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_split=2,
            class_weight='balanced',
            random_state=42
        ))
    ])
    rf_pipe.fit(X_tr, y_tr)
    print(f'\nRandom Forest accuracy ({("train" if len(X)<20 else "test")}) : {rf_pipe.score(X_te, y_te):.2%}')

    # XGBoost (use_label_encoder removed — deprecated since XGBoost 1.6)
    xgb_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', xgb.XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            eval_metric='mlogloss',
            random_state=42
        ))
    ])
    xgb_pipe.fit(X_tr, y_tr)
    print(f'XGBoost accuracy ({("train" if len(X)<20 else "test")})       : {xgb_pipe.score(X_te, y_te):.2%}')

    # Cross-validation — this is the honest number to report
    from collections import Counter
    class_counts = Counter(y_enc)
    min_class_n  = min(class_counts.values())
    n_splits     = min(5, min_class_n)

    print(f'\nClass sizes: {dict(class_counts)}')
    print(f'CV folds   : {n_splits}')

    if n_splits < 2:
        print('⚠️  Smallest class has only 1 sample — cannot cross-validate')
        rf_cv  = np.array([rf_pipe.score(X_te, y_te)])
        xgb_cv = np.array([xgb_pipe.score(X_te, y_te)])
    else:
        cv     = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        rf_cv  = cross_val_score(rf_pipe,  X, y_enc, cv=cv, scoring='f1_weighted')
        xgb_cv = cross_val_score(xgb_pipe, X, y_enc, cv=cv, scoring='f1_weighted')

    print(f'\nCV F1 weighted (RF)  : {rf_cv.mean():.3f} ± {rf_cv.std():.3f}')
    print(f'CV F1 weighted (XGB) : {xgb_cv.mean():.3f} ± {xgb_cv.std():.3f}')
    print(f'  ↑ Report THIS number (cross-validated), not the test-set accuracy')

    # Ensemble probabilities on X_te (equals X when n<20 — used for quick printout only)
    rf_probs  = rf_pipe.predict_proba(X_te)
    xgb_probs = xgb_pipe.predict_proba(X_te)
    ens_probs = (rf_probs + xgb_probs) / 2.0
    ens_preds = np.argmax(ens_probs, axis=1)

    # ── LOOCV for honest confusion matrix / per-class metrics ────────
    # When a fold is missing a class (e.g. only 1 pseudo-label sample),
    # probabilities are aligned to the full class set by integer index.
    from sklearn.model_selection import LeaveOneOut

    def _align_proba(pipe, X_te_fold, n_classes):
        """Map predict_proba output to full n_classes columns using clf.classes_."""
        proba_fold = pipe.predict_proba(X_te_fold)
        full = np.zeros((proba_fold.shape[0], n_classes))
        for local_i, global_cls in enumerate(pipe.named_steps['clf'].classes_):
            full[:, global_cls] = proba_fold[:, local_i]
        return full

    if len(X) < 20:
        loo           = LeaveOneOut()
        loo_preds_arr = np.zeros(len(X), dtype=int)
        loo_probs_arr = np.zeros((len(X), len(le.classes_)))
        for tr_idx, te_idx in loo.split(X):
            rf_pipe.fit(X[tr_idx], y_enc[tr_idx])
            xgb_pipe.fit(X[tr_idx], y_enc[tr_idx])
            rp = _align_proba(rf_pipe,  X[te_idx], len(le.classes_))
            xp = _align_proba(xgb_pipe, X[te_idx], len(le.classes_))
            ep = (rp + xp) / 2.0
            loo_preds_arr[te_idx] = np.argmax(ep, axis=1)
            loo_probs_arr[te_idx] = ep
        # Refit on all data so the saved model uses everything
        rf_pipe.fit(X_tr, y_tr)
        xgb_pipe.fit(X_tr, y_tr)
        ens_preds_display = loo_preds_arr
        ens_probs_display = loo_probs_arr
        y_te_display      = y_enc
        eval_label        = 'LOOCV'
        loo_acc = np.mean(loo_preds_arr == y_enc)
        loo_f1  = f1_score(y_enc, loo_preds_arr, average='weighted', zero_division=0)
        print(f'\nLOOCV accuracy     : {loo_acc:.2%}')
        print(f'LOOCV F1 weighted  : {loo_f1:.3f}')
        print('  ↑ Use THESE numbers — confusion matrix and per-class metrics are LOOCV')
    else:
        ens_preds_display = ens_preds
        ens_probs_display = ens_probs
        y_te_display      = y_te
        eval_label        = 'Test set (20%)'
    print(f'\nEvaluation method for plots: {eval_label}')

    # Save
    with open(os.path.join(MODELS_DIR, 'random_forest.pkl'),  'wb') as f: pickle.dump(rf_pipe,  f)
    with open(os.path.join(MODELS_DIR, 'xgboost_model.pkl'), 'wb') as f: pickle.dump(xgb_pipe, f)
    with open(os.path.join(MODELS_DIR, 'label_encoder.pkl'), 'wb') as f: pickle.dump(le,       f)
    with open(os.path.join(MODELS_DIR, 'feature_cols.pkl'),  'wb') as f: pickle.dump(feat_cols,f)
    print('\nAll models saved!')

## 7. Apply Ensemble to Science Data

In [20]:
if ml_available:
    X_sci     = science_df[feat_cols].fillna(0).values
    rf_p      = rf_pipe.predict_proba(X_sci)
    xgb_p     = xgb_pipe.predict_proba(X_sci)
    ens_p     = (rf_p + xgb_p) / 2.0
    ens_pred  = le.inverse_transform(np.argmax(ens_p, axis=1))
    ens_conf  = ens_p.max(axis=1)

    science_df['ml_label']      = ens_pred
    science_df['ml_confidence'] = ens_conf
    print('Ensemble classification:')
    print(science_df['ml_label'].value_counts())
else:
    science_df['ml_label']      = science_df['rule_label']
    science_df['ml_confidence'] = science_df['rule_confidence']

science_df['final_label']      = science_df['ml_label']
science_df['final_confidence'] = science_df['ml_confidence']
science_df.to_csv('../outputs/results.csv', index=False)
print()
print(science_df[['tic_id','final_label','final_confidence',
                  'snr']].to_string())

Ensemble classification:
ml_label
planet              11
eclipsing_binary     3
Name: count, dtype: int64

       tic_id       final_label  final_confidence        snr
0   149603524  eclipsing_binary          0.584648   6.501278
1   229742722            planet          0.707221   4.431758
2   237201858            planet          0.677221   4.609320
3   260647166  eclipsing_binary          0.665781   8.407794
4   261136641            planet          0.803894   5.004373
5   261136679            planet          0.766394  28.247613
6   261136765            planet          0.558155   4.980574
7   261139167            planet          0.734913   5.009299
8   261155555            planet          0.540318  35.553242
9   271893367            planet          0.597340  11.333272
10  279741379  eclipsing_binary          0.575284  16.047355
11  350618622            planet          0.740115   3.824662
12  441075486            planet          0.760821   9.213031
13   55525572            planet        

## 8. Confusion Matrix + Feature Importance

In [ ]:
if ml_available:
    from sklearn.metrics import classification_report, precision_recall_fscore_support
    from sklearn.model_selection import learning_curve

    fig = plt.figure(figsize=(20, 16))
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.4)

    # ── Panel A: Normalised Confusion Matrix ─────────────────────────
    ax_cm = fig.add_subplot(gs[0, 0])
    cm_norm = confusion_matrix(y_te_display, ens_preds_display, normalize='true')
    im = ax_cm.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax_cm, fraction=0.046, pad=0.04)
    for i in range(cm_norm.shape[0]):
        for j in range(cm_norm.shape[1]):
            ax_cm.text(j, i, f'{cm_norm[i,j]:.0%}',
                       ha='center', va='center', fontsize=11, fontweight='bold',
                       color='white' if cm_norm[i,j] > 0.5 else 'black')
    ax_cm.set_xticks(range(len(le.classes_)))
    ax_cm.set_yticks(range(len(le.classes_)))
    ax_cm.set_xticklabels([c.replace('_','\n') for c in le.classes_], fontsize=9)
    ax_cm.set_yticklabels([c.replace('_','\n') for c in le.classes_], fontsize=9)
    ax_cm.set_xlabel('Predicted', fontsize=10)
    ax_cm.set_ylabel('True', fontsize=10)
    ax_cm.set_title(f'(A) Confusion Matrix\n(Ensemble, {eval_label}, Normalised)', fontweight='bold')

    # ── Panel B: CV F1 Scores — RF vs XGB vs Ensemble ───────────────
    ax_cv = fig.add_subplot(gs[0, 1])
    # Compute ensemble CV F1
    ens_cv_scores = []
    cv_splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    for tr_idx, te_idx in cv_splitter.split(X, y_enc):
        rf_pipe.fit(X[tr_idx], y_enc[tr_idx])
        xgb_pipe.fit(X[tr_idx], y_enc[tr_idx])
        rp = rf_pipe.predict_proba(X[te_idx])
        xp = xgb_pipe.predict_proba(X[te_idx])
        ep = np.argmax((rp + xp) / 2, axis=1)
        ens_cv_scores.append(f1_score(y_enc[te_idx], ep, average='weighted', zero_division=0))
    ens_cv = np.array(ens_cv_scores)
    # Refit on full training data
    rf_pipe.fit(X_tr, y_tr)
    xgb_pipe.fit(X_tr, y_tr)

    models_cv = {'Random\nForest': rf_cv, 'XGBoost': xgb_cv, 'Ensemble\n(RF+XGB)': ens_cv}
    colors_cv  = ['#3498db', '#e67e22', '#2ecc71']
    x_pos = np.arange(len(models_cv))
    bars  = ax_cv.bar(x_pos, [v.mean() for v in models_cv.values()],
                      color=colors_cv, edgecolor='white', linewidth=0.8, width=0.5)
    ax_cv.errorbar(x_pos, [v.mean() for v in models_cv.values()],
                   yerr=[v.std() for v in models_cv.values()],
                   fmt='none', color='black', capsize=6, linewidth=2)
    for bar, vals in zip(bars, models_cv.values()):
        ax_cv.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                   f'{vals.mean():.3f}', ha='center', va='bottom',
                   fontsize=10, fontweight='bold')
    ax_cv.set_xticks(x_pos)
    ax_cv.set_xticklabels(list(models_cv.keys()), fontsize=10)
    ax_cv.set_ylabel('Weighted F1 Score', fontsize=10)
    ax_cv.set_ylim(0, 1.15)
    ax_cv.set_title(f'(B) Cross-Validated F1 Score\n({n_splits}-fold Stratified CV)', fontweight='bold')
    ax_cv.axhline(1.0, color='gray', linestyle='--', lw=0.8, alpha=0.5)

    # ── Panel C: Per-class Precision / Recall / F1 ──────────────────
    ax_pr = fig.add_subplot(gs[0, 2])
    prec, rec, f1s, _ = precision_recall_fscore_support(
        y_te_display, ens_preds_display, labels=range(len(le.classes_)), zero_division=0
    )
    x_cls   = np.arange(len(le.classes_))
    w       = 0.25
    ax_pr.bar(x_cls - w, prec, w, label='Precision', color='#3498db', edgecolor='white')
    ax_pr.bar(x_cls,     rec,  w, label='Recall',    color='#e74c3c', edgecolor='white')
    ax_pr.bar(x_cls + w, f1s,  w, label='F1',        color='#2ecc71', edgecolor='white')
    ax_pr.set_xticks(x_cls)
    ax_pr.set_xticklabels([c.replace('_','\n') for c in le.classes_], fontsize=9)
    ax_pr.set_ylabel('Score', fontsize=10)
    ax_pr.set_ylim(0, 1.2)
    ax_pr.set_title(f'(C) Per-class Precision / Recall / F1\n(Ensemble, {eval_label})', fontweight='bold')
    ax_pr.legend(fontsize=9)
    for i, (p, r, f) in enumerate(zip(prec, rec, f1s)):
        ax_pr.text(i - w, p + 0.03, f'{p:.2f}', ha='center', fontsize=8)
        ax_pr.text(i,     r + 0.03, f'{r:.2f}', ha='center', fontsize=8)
        ax_pr.text(i + w, f + 0.03, f'{f:.2f}', ha='center', fontsize=8)

    # ── Panel D: Feature Importance (RF) — top 15 ───────────────────
    ax_fi = fig.add_subplot(gs[1, :2])
    imp_df = pd.DataFrame({
        'feature'   : feat_cols,
        'importance': rf_pipe.named_steps['clf'].feature_importances_
    }).sort_values('importance', ascending=True).tail(15)

    def feat_color(f):
        if any(k in f for k in ['tls', 'sde', 'fap', 'rp_rs', 'odd_even']): return '#e74c3c'
        if any(k in f for k in ['cdpp']):                                      return '#9b59b6'
        if any(k in f for k in ['acf', 'ls_']):                               return '#f39c12'
        if any(k in f for k in ['bic', 'delta']):                             return '#1abc9c'
        return '#3498db'

    bar_colors = [feat_color(f) for f in imp_df['feature']]
    ax_fi.barh(imp_df['feature'], imp_df['importance'],
               color=bar_colors, edgecolor='white', linewidth=0.5, height=0.7)
    ax_fi.set_xlabel('Feature Importance (Mean Decrease Impurity)', fontsize=10)
    ax_fi.set_title('(D) Top 15 Feature Importances — Random Forest', fontweight='bold')

    legend_items = [
        mpatches.Patch(color='#e74c3c', label='TLS features'),
        mpatches.Patch(color='#9b59b6', label='CDPP noise'),
        mpatches.Patch(color='#f39c12', label='ACF / LS periodicity'),
        mpatches.Patch(color='#1abc9c', label='BIC score'),
        mpatches.Patch(color='#3498db', label='Classical BLS / shape'),
    ]
    ax_fi.legend(handles=legend_items, fontsize=8, loc='lower right')

    # ── Panel E: Learning Curve ──────────────────────────────────────
    ax_lc = fig.add_subplot(gs[1, 2])
    if len(X) >= 8:
        train_sizes, train_scores, val_scores = learning_curve(
            rf_pipe, X, y_enc, cv=min(3, n_splits),
            scoring='f1_weighted', n_jobs=-1,
            train_sizes=np.linspace(0.3, 1.0, 6)
        )
        ax_lc.plot(train_sizes, train_scores.mean(axis=1), 'o-',
                   color='#3498db', label='Train F1', lw=2)
        ax_lc.fill_between(train_sizes,
                            train_scores.mean(axis=1) - train_scores.std(axis=1),
                            train_scores.mean(axis=1) + train_scores.std(axis=1),
                            alpha=0.2, color='#3498db')
        ax_lc.plot(train_sizes, val_scores.mean(axis=1), 'o-',
                   color='#e74c3c', label='CV F1', lw=2)
        ax_lc.fill_between(train_sizes,
                            val_scores.mean(axis=1) - val_scores.std(axis=1),
                            val_scores.mean(axis=1) + val_scores.std(axis=1),
                            alpha=0.2, color='#e74c3c')
        ax_lc.set_xlabel('Training Samples', fontsize=10)
        ax_lc.set_ylabel('Weighted F1 Score', fontsize=10)
        ax_lc.set_title('(E) Learning Curve\n(Random Forest)', fontweight='bold')
        ax_lc.legend(fontsize=9)
        ax_lc.set_ylim(0, 1.1)
    else:
        ax_lc.text(0.5, 0.5, 'Need ≥8 samples\nfor learning curve',
                   ha='center', va='center', transform=ax_lc.transAxes, fontsize=11)
        ax_lc.set_title('(E) Learning Curve', fontweight='bold')

    # ── Panel F: Ensemble confidence by class (uses LOOCV probs when n<20) ──
    ax_conf = fig.add_subplot(gs[2, :])
    max_conf   = ens_probs_display.max(axis=1)
    pred_class = le.inverse_transform(np.argmax(ens_probs_display, axis=1))
    true_class = le.inverse_transform(y_te_display)
    correct    = pred_class == true_class

    cls_palette = {'planet': '#2ecc71', 'eclipsing_binary': '#e74c3c',
                   'false_positive': '#f39c12', 'planet_candidate': '#27ae60'}

    for cls in le.classes_:
        mask_cls = true_class == cls
        col = cls_palette.get(cls, '#3498db')
        ax_conf.scatter(
            np.where(mask_cls)[0], max_conf[mask_cls],
            c=[col if c else '#bdc3c7' for c in correct[mask_cls]],
            s=120, edgecolors=col, linewidths=1.5,
            label=f'{cls} (✓=filled, ✗=gray)', zorder=5, alpha=0.85
        )
    ax_conf.axhline(0.5, color='red', linestyle='--', lw=1.5, label='50% confidence')
    ax_conf.axhline(0.7, color='orange', linestyle=':', lw=1.5, label='70% confidence')
    ax_conf.set_xlabel('Sample Index', fontsize=10)
    ax_conf.set_ylabel('Max Class Probability (Confidence)', fontsize=10)
    ax_conf.set_title(f'(F) Ensemble Prediction Confidence per Sample ({eval_label})\n'
                      '(filled = correct prediction, gray = wrong)', fontweight='bold')
    ax_conf.set_ylim(0, 1.1)
    ax_conf.legend(fontsize=9, ncol=3)

    # ── Suptitle with key metric ─────────────────────────────────────
    best_cv = max(rf_cv.mean(), xgb_cv.mean(), ens_cv.mean())
    plt.suptitle(
        f'Model Training — RF + XGBoost Ensemble  |  '
        f'Best CV F1 = {best_cv:.3f}  |  '
        f'Dataset: {len(X)} labeled samples  |  Classes: {list(le.classes_)}',
        fontsize=12, fontweight='bold', y=1.01
    )

    plt.savefig('../outputs/plots/model_training_full.png', dpi=200, bbox_inches='tight')
    plt.show()
    print(f'\nModel training visualization saved → outputs/plots/model_training_full.png')
    print(f'\nClassification Report (Ensemble, {eval_label}):')
    print(classification_report(y_te_display, ens_preds_display, target_names=le.classes_, zero_division=0))

---
## ✅ Done!
**Fixes applied:**
- `delta_bic` added to `ALL_FEATURES` / `FEATURE_COLS` (was previously a special-cased append)
- Removed deprecated `use_label_encoder=False` from XGBoost (dropped in XGBoost ≥1.6)
- **4-class ML**: labeled_df only has planet / eclipsing_binary, so high-confidence rule-based predictions on science data are used as pseudo-labels to add false_positive and planet_candidate examples
- **LOOCV** used when n < 20: Panels A, C, F and the classification report show honest held-out predictions; probability alignment handles folds where a rare class is left out

**Next → 06_classification.ipynb**